# Retail Store Performance: EDA and Hypothesis Testing

**Academic project.** Exploratory analysis and inferential tests on a 400-store retail dataset to find what drives unit sales: shelf location, market, price, and advertising.

> **Dataset not included (course-provided).** This notebook uses a 400-store retail file supplied for a university course (`2nd_assignment.csv`, a version of the Carseats data from *An Introduction to Statistical Learning*). The course file is not redistributed, so the notebook is published **without outputs**. Numbers quoted in the markdown come from the original course run. To run it, place the file in `data/`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

data = pd.read_csv(DATA_DIR / '2nd_assignment.csv')

quantitative_vars = ['Sales', 'CompPrice', 'Income', 'Advertising', 'Population', 'Price', 'Age', 'Education']
categorical_vars = ['ShelveLoc', 'Urban', 'US']
data.head()

## Phase 1: Missing values and duplicates

Quantitative variables are imputed with the median (robust to skew) and categorical variables with the mode.

In [ ]:
def impute_missing_values(df, quant_vars, cat_vars):
    df_clean = df.copy()
    for var in quant_vars:
        df_clean[var] = df_clean[var].fillna(df_clean[var].median())
    for var in cat_vars:
        df_clean[var] = df_clean[var].fillna(df_clean[var].mode()[0])
    return df_clean

print('Missing values before imputation:')
print(data[quantitative_vars + categorical_vars].isna().sum())

data_imputed = impute_missing_values(data, quantitative_vars, categorical_vars)
data_clean = data_imputed.drop_duplicates()
print(f'Dataset cleaned. Final shape: {data_clean.shape}')

# Saved for the regression notebook (03_ols_regression.ipynb)
data_clean.to_csv(DATA_DIR / '2nd_assignment_clean.csv', index=False)

## Phase 2: Extreme outliers (3 x IQR rule)

Outliers are flagged but kept, because in retail they usually reflect real high-impact variance rather than data errors.

In [ ]:
def detect_extreme_outliers(df, variable):
    q1, q3 = df[variable].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    return df[(df[variable] < lower) | (df[variable] > upper)]

print('--- EXTREME OUTLIERS DETECTED ---')
for var in quantitative_vars:
    outliers = detect_extreme_outliers(data_clean, var)
    if not outliers.empty:
        print(f'[{var}]: {len(outliers)} extreme outliers')

## Phase 3: Exploratory analysis

Original course run: `Price` correlates 0.58 with competitor price (`CompPrice`) and -0.45 with `Sales`.

In [ ]:
corr_matrix = data_clean[quantitative_vars].corr().round(2)
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, vmin=-1, vmax=1, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Retail Variables')
plt.show()

In [ ]:
def analyze_categorical_relationships(df, cat_vars):
    for i in range(len(cat_vars)):
        for j in range(i + 1, len(cat_vars)):
            var1, var2 = cat_vars[i], cat_vars[j]
            crosstab = pd.crosstab(df[var1], df[var2], normalize='index').round(2)
            print(f'\n--- Distribution: {var1} vs {var2} ---')
            print(crosstab)

analyze_categorical_relationships(data_clean, categorical_vars)

In [ ]:
def plot_business_impact(df, cate_var, target_var):
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=cate_var, y=target_var, data=df, order=sorted(df[cate_var].unique()), color='#7BA7D9')
    plt.title(f'Impact of {cate_var} on {target_var}')
    plt.show()

plot_business_impact(data_clean, 'ShelveLoc', 'Sales')
plot_business_impact(data_clean, 'US', 'Sales')

## Phase 4: Hypothesis testing

- **Chi-square tests** check whether pairs of categorical variables are independent.
- **ANOVA F-tests** (one-way, via OLS) check whether each categorical variable explains variance in each quantitative variable.

Original course run: every chi-square pair was independent (for example ShelveLoc vs Urban, p = 0.2335). ANOVA found ShelveLoc to Sales (p < 0.0001), US to Sales (p = 0.0004), and US to Advertising (p < 0.0001).

In [ ]:
def chi_square_test(dataset, column1, column2):
    pivot_table = pd.crosstab(dataset[column1], dataset[column2])
    chisq, pvalue, dof, expected = stats.chi2_contingency(pivot_table)
    return pvalue

print('--- CHI-SQUARE TESTS OF INDEPENDENCE ---')
for i in range(len(categorical_vars)):
    for j in range(i + 1, len(categorical_vars)):
        var1, var2 = categorical_vars[i], categorical_vars[j]
        pvalue = chi_square_test(data_clean, var1, var2)
        label = 'SIGNIFICANT' if pvalue < 0.05 else 'INDEPENDENT'
        print(f'[{label}] {var1} & {var2} | p-value: {pvalue:.4f}')

In [ ]:
def anova_f_test(dataset, cont_var, cate_var):
    model = smf.ols(f'{cont_var} ~ {cate_var}', data=dataset).fit()
    anova_table = sm.stats.anova_lm(model)
    return anova_table['PR(>F)'].iloc[0]

print('--- ANOVA F-TESTS (p < 0.05 only) ---')
for cont_var in quantitative_vars:
    for cate_var in categorical_vars:
        pvalue = anova_f_test(data_clean, cont_var, cate_var)
        if pvalue < 0.05:
            print(f'[DRIVER] {cate_var} -> {cont_var} | p-value: {pvalue:.4f}')

## Takeaways (original course run)

- Shelf location is the strongest operational driver of sales.
- US stores sell more and also spend more on advertising, which partly explains their advantage.
- Shelf placement is independent of whether a store is urban or in the US, so it is a lever management controls directly.
- Next: quantify these effects with regression (`03_ols_regression.ipynb`).